# Data Pipeline — Satellite Land-Use Classifier

This notebook documents the data pipeline for the EuroSAT dataset.

## Pipeline Overview

1. **Dataset loading** — `EuroSATDataset` walks class-named subdirectories and maps folder names to integer labels.
2. **Image preprocessing** — Images are resized to 224×224, converted to tensors, and normalized with ImageNet statistics. Training applies random flip, rotation, and color jitter.
3. **Spatial block split** — Images are grouped into spatial blocks (100 images per block by sorted filename order) and entire blocks are assigned to train/val/test splits. This prevents geographic leakage where nearby patches from the same Sentinel-2 scene appear across splits.
4. **Visualization** — Sample batch display, 5 samples per class, and class distribution.

## Spatial Block Split

The primary data pipeline uses a **spatial block split** (`create_eurosat_datasets` in `src/dataset.py`):
- Images within each class are sorted by filename (deterministic order).
- Consecutive images are grouped into blocks of 100.
- Blocks are shuffled and assigned proportionally (70/15/15) to train/val/test.
- Images in the same block never cross splits.

A random stratified split is also available as `create_eurosat_datasets_stratified()` for the spatial leakage experiment.

## Dataset Info
- **EuroSAT**: 27,000 RGB 64×64 Sentinel-2 tiles, 10 balanced classes (2,700 each).
- **UC Merced**: 2,100 RGB 256×256 aerial images, 21 classes — used for zero-shot cross-domain evaluation.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import EUROSAT_ROOT, CLASS_NAMES, IMAGE_SIZE, BATCH_SIZE, SEED
from src.dataloader import build_dataloaders
from src.dataset import _discover_paths
from src.transforms import get_eval_transforms
from notebooks.visualize_batch import (
    denormalize,
    visualize_five_per_class,
    visualize_class_distribution,
)
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from torchvision.utils import make_grid
import torch
from PIL import Image
import random

In [ ]:
# --- 1. Sample batch visualization ---
loaders = build_dataloaders(
    root=EUROSAT_ROOT,
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=0,
    pin_memory=False,
    seed=SEED,
)

images, labels = next(iter(loaders["train"]))
print(f"Batch shape: {images.shape}")
print(f"Labels: {[CLASS_NAMES[l] for l in labels.tolist()]}")

nrow = 8
grid = make_grid(images, nrow=nrow, padding=2, normalize=False)
grid_img = denormalize(grid)

fig, ax = plt.subplots(figsize=(16, 8))
ax.imshow(grid_img)
ax.axis("off")
ax.set_title("EuroSAT Training Batch (denormalised)", fontsize=14)
for i in range(len(labels)):
    row = i // nrow
    col = i % nrow
    x = col * (IMAGE_SIZE[1] + 4) + IMAGE_SIZE[1] // 2
    y = row * (IMAGE_SIZE[0] + 4) + IMAGE_SIZE[0] + 8
    ax.text(x, y, CLASS_NAMES[labels[i]], ha="center", va="top", fontsize=6)
plt.show()

In [ ]:
# --- 2. Five samples per class ---
reports_dir = PROJECT_ROOT / "reports"
visualize_five_per_class(reports_dir)

In [ ]:
# --- 3. Class distribution ---
visualize_class_distribution(reports_dir)

## Spatial Block Split — Verification

The cells below verify that the spatial block split prevents geographic leakage by checking that no image path appears in more than one split.

In [ ]:
# --- 4. Verify spatial block split ---
from src.dataset import create_spatial_datasets

ds = create_spatial_datasets(
    root=EUROSAT_ROOT,
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    block_size=100,
    seed=SEED,
)

train_set = set(ds["train"].file_paths)
val_set = set(ds["val"].file_paths)
test_set = set(ds["test"].file_paths)

overlap_tv = train_set & val_set
overlap_tt = train_set & test_set
overlap_vt = val_set & test_set

print(f"Train/val overlap: {len(overlap_tv)} images")
print(f"Train/test overlap: {len(overlap_tt)} images")
print(f"Val/test overlap: {len(overlap_vt)} images")
print(f"All overlaps zero: {not (overlap_tv or overlap_tt or overlap_vt)}")
print(f"Total images: {len(ds['train']) + len(ds['val']) + len(ds['test'])}")
print(f"Split sizes — Train: {len(ds['train'])}, Val: {len(ds['val'])}, Test: {len(ds['test'])}")